# US-04-02 — Restauration CPU indépendante — PatchCore Bottle (EXP-002)

**Objectif :** restaurer le checkpoint complet de l’expérience `EXP-002` dans **une nouvelle session Kaggle CPU**, sans reconstruire la Memory Bank ni réentraîner le backbone. Vérifier que le score et la carte d’anomalie correspondent à la prédiction de référence GPU.

Le **code source de l'expérience GPU** est figé par le tag `exp/patchcore-bottle-exp002` (commit `4664a8e8c016ea864c65b94f959456a50afbb7a4`). Le notebook de restauration est un livrable complémentaire, à versionner séparément après validation.

## 1 — Localiser et vérifier les artefacts de l'expérience GPU

Nous devons retrouver le checkpoint, le rapport JSON et la prédiction GPU. Un fichier `.npz` est une archive NumPy : on le lit directement, sans le décompresser.

In [ ]:
from pathlib import Path
import hashlib
import json

EXPECTED_COMMIT = "4664a8e8c016ea864c65b94f959456a50afbb7a4"
EXPECTED_TAG = "exp/patchcore-bottle-exp002"
EXPECTED_CHECKPOINT_SHA256 = (
    "dcf0f8df3dc9b4be9ee1bcb897d3ec6f24a3191240dcfd4e68afd598acd77cde"
)
SOURCE_KAGGLE_URL = (
    "https://www.kaggle.com/code/mouadfekkak/14-patchcore-bottle-memory-bank"
    "?scriptVersionId=369295449"
)

input_root = Path("/kaggle/input")
assert input_root.is_dir(), "Ce notebook doit être exécuté sur Kaggle."

# Uniquement le checkpoint de la banque EXP-002 (pas le smoke test EXP-001).
checkpoints = sorted(
    path for path in input_root.rglob("patchcore_model.pt")
    if path.parent.name == "EXP-002" and path.parent.parent.name == "bottle"
)
print("Checkpoints EXP-002 trouvés:", len(checkpoints))
for path in checkpoints:
    print(" -", path)
assert len(checkpoints) == 1, (
    "Ajouter exactement une fois les Outputs du notebook GPU EXP-002 "
    "dans Add Input > Notebook Output."
)

checkpoint_path = checkpoints[0]
artifact_dir = checkpoint_path.parent
report_path = artifact_dir / "memory_bank_report.json"
reference_path = artifact_dir / "reference_prediction.npz"

assert report_path.is_file(), f"Rapport manquant : {report_path}"
assert reference_path.is_file(), f"Référence manquante : {reference_path}"

report = json.loads(report_path.read_text(encoding="utf-8"))
actual_checkpoint_sha256 = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()

assert actual_checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256
assert report["checkpoint_sha256"] == actual_checkpoint_sha256
assert report["experiment_id"] == "EXP-002"
assert report["git_commit"] == EXPECTED_COMMIT
assert report["git_tag"] == EXPECTED_TAG
assert report["train_image_count"] == 167
assert report["embedding_shape"] == [171008, 1536]
assert report["memory_bank_shape"] == [1710, 1536]

print("\n=== SOURCE GPU VALIDÉE ===")
print("Run:", report["run_id"])
print("Commit:", report["git_commit"])
print("Tag:", report["git_tag"])
print("Checkpoint SHA-256:", actual_checkpoint_sha256)
print("Memory Bank:", report["memory_bank_shape"])
print("Image de validation:", report["validation_image"])
print("Source GPU artifacts: OK")

## 2 — Installer les dépendances de restauration

On réutilise l'implémentation Anomalib de l'US-04-01, avec les versions critiques de la session GPU. Les calculs seront **forcés sur CPU**.

In [ ]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "anomalib==2.7.0", "timm==1.0.30", "lightning==2.6.6"
    ],
    check=True
)
print("Dépendances de restauration installées : OK")

## 3 — Contrôler l'environnement CPU

Un package PyTorch marqué `+cu130` peut exister même sans GPU actif. La compatibilité est vérifiée sur sa version de base et toutes les prédictions sont calculées sur `cpu`.

In [ ]:
import importlib.metadata as metadata
from packaging.version import Version
import torch
import torchvision

DEVICE = torch.device("cpu")
print("=== ENVIRONNEMENT DE RESTAURATION ===")
print("Python:", sys.version.split()[0])
print("CUDA disponible:", torch.cuda.is_available())
print("Calcul forcé sur:", DEVICE)

actual_versions = {}
for package, expected in report["versions"].items():
    actual = metadata.version(package)
    actual_versions[package] = actual
    if package in ("torch", "torchvision"):
        compatible = Version(actual).base_version == Version(expected).base_version
    else:
        compatible = actual == expected
    print(f"{package}: {actual} (source GPU: {expected}) -> "
          f"{'OK' if compatible else 'INCOMPATIBLE'}")
    assert compatible, f"Version incompatible: {package}"

print("Versions compatibles et exécution CPU : OK")

## 4 — Recharger le code Git exact et la même image de validation

Le split **Validation** contient 42 images normales. L'image de référence est indiquée dans le rapport GPU ; le split **Test** reste totalement inutilisé.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MRdrissi/industrial-defect-ai.git"
REPO_DIR = Path("/kaggle/working/industrial-defect-ai")

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "--tags"], check=True)
tagged_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", f"refs/tags/{EXPECTED_TAG}^{{commit}}"],
    text=True
).strip()
assert tagged_commit == EXPECTED_COMMIT

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", "--detach", EXPECTED_COMMIT],
    check=True
)
source_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert source_commit == EXPECTED_COMMIT

sys.path.insert(0, str(REPO_DIR))
from src.data.mvtec_dataset import MVTecDataset
from src.data.transforms import MVTecTransform

config_path = REPO_DIR / "experiments/configs/patchcore_bottle.json"
config_sha256 = hashlib.sha256(config_path.read_bytes()).hexdigest()
assert config_sha256 == report["config_sha256"]
config = json.loads(config_path.read_text(encoding="utf-8"))
assert config["experiment_id"] == "EXP-002"
assert config["category"] == "bottle"

manifest = REPO_DIR / config["manifest"]
manifest_sha256 = hashlib.sha256(manifest.read_bytes()).hexdigest()
assert manifest_sha256 == report["manifest_sha256"]

# Retrouver la racine MVTec sans supposer le chemin exact de l'Input.
dataset_root = None
for current, dirs, _ in os.walk(input_root):
    candidate = Path(current)
    if "bottle" in dirs and (candidate / "bottle/train/good").is_dir():
        dataset_root = candidate
        break
assert dataset_root is not None, "Ajouter ipythonx/mvtec-ad dans Add Input > Datasets."

val_dataset = MVTecDataset(
    dataset_root=dataset_root,
    manifest_path=manifest,
    category="bottle",
    split="val",
    transform=MVTecTransform(
        size=tuple(config["transform"]["size"]),
        profile=config["transform"]["profile"]
    )
)
assert len(val_dataset) == config["inference"]["expected_samples"] == 42

target_path = report["validation_image"]
assert target_path == config["inference"]["image_path"]
assert target_path not in report["train_images"]
matching_indices = [
    i for i, row in enumerate(val_dataset.samples)
    if row["image_path"] == target_path
]
assert len(matching_indices) == 1
sample = val_dataset[matching_indices[0]]
assert int(sample["label"]) == 0
assert tuple(sample["image"].shape) == (3, 256, 256)

print("=== DONNÉES IDENTIQUES ===")
print("Commit source:", source_commit)
print("Config SHA-256:", config_sha256)
print("Manifest SHA-256:", manifest_sha256)
print("Dataset:", dataset_root)
print("Validation (42 normales), image:", sample["image_path"])
print("Code, config, manifeste et validation : OK")

## 5 — Restaurer le modèle et effectuer l'inférence CPU

**Aucune nouvelle extraction Train, aucun KCenterGreedy, aucun réentraînement.** Le checkpoint suffit pour retrouver le backbone et la banque des **1 710** vecteurs.

In [ ]:
import numpy as np
from anomalib.models.image.patchcore.torch_model import PatchcoreModel

saved = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
assert saved["backbone"] == config["model"]["backbone"] == "wide_resnet50_2"
assert saved["layers"] == config["model"]["layers"] == ["layer2", "layer3"]
assert saved["num_neighbors"] == config["model"]["num_neighbors"] == 1

restored_model = PatchcoreModel(
    backbone=saved["backbone"],
    layers=saved["layers"],
    pre_trained=False,
    num_neighbors=saved["num_neighbors"]
).to(DEVICE)
restored_model.load_state_dict(saved["model_state_dict"], strict=True)
restored_model.feature_extractor.requires_grad_(False)
restored_model.eval()

assert tuple(restored_model.memory_bank.shape) == tuple(report["memory_bank_shape"])
assert restored_model.memory_bank.device.type == "cpu"
assert torch.isfinite(restored_model.memory_bank).all()
assert all(not p.requires_grad for p in restored_model.feature_extractor.parameters())

with np.load(reference_path, allow_pickle=False) as reference:
    expected_score = torch.from_numpy(reference["pred_score"]).to(DEVICE)
    expected_map = torch.from_numpy(reference["anomaly_map"]).to(DEVICE)

with torch.inference_mode():
    prediction = restored_model(sample["image"].unsqueeze(0).to(DEVICE))

actual_score = prediction.pred_score
actual_map = prediction.anomaly_map

assert actual_score.shape == expected_score.shape
assert actual_map.shape == expected_map.shape == (1, 1, 256, 256)
assert torch.isfinite(actual_score).all()
assert torch.isfinite(actual_map).all()

print("=== INFÉRENCE CPU ===")
print("Memory Bank restaurée:", tuple(restored_model.memory_bank.shape))
print("Score GPU de référence:", expected_score.item())
print("Score CPU restauré:", actual_score.item())
print("Carte d'anomalie:", tuple(actual_map.shape))
print("Restauration et inférence CPU : OK")

## 6 — Comparer CPU et GPU selon des tolérances explicites

Les opérations flottantes CPU/GPU peuvent différer légèrement : on valide une proximité numérique, pas une identité bit à bit.

In [ ]:
SCORE_ABS_TOL = 1e-3
MAP_ABS_TOL = 1e-3
MAP_RANGE_REL_TOL = 1e-4

score_abs_diff = (actual_score - expected_score).abs().max().item()
map_abs_error = (actual_map - expected_map).abs()
map_max_diff = map_abs_error.max().item()
map_mean_diff = map_abs_error.mean().item()
map_range = (expected_map.max() - expected_map.min()).item()
assert map_range > 0
map_relative_range_error = map_max_diff / map_range

print("=== COHÉRENCE GPU / CPU ===")
print("Écart absolu score:", score_abs_diff, "(tolérance:", SCORE_ABS_TOL, ")")
print("Écart max carte:", map_max_diff, "(tolérance:", MAP_ABS_TOL, ")")
print("Écart moyen carte:", map_mean_diff)
print("Écart max / étendue carte:", map_relative_range_error,
      "(tolérance:", MAP_RANGE_REL_TOL, ")")

assert score_abs_diff <= SCORE_ABS_TOL, "Écart de score trop élevé"
assert map_max_diff <= MAP_ABS_TOL, "Écart maximal de carte trop élevé"
assert map_relative_range_error <= MAP_RANGE_REL_TOL, "Écart relatif trop élevé"

print("\nINDEPENDENT CPU RESTORATION: PASS")

## 7 — Enregistrer le rapport de restauration

Le rapport JSON sera téléchargé depuis les **Outputs de ce notebook CPU**, puis placé dans `docs/data/us-04-02/`. Ne pas ajouter le checkpoint `.pt` à Git.

In [ ]:
RESTORE_DIR = Path("/kaggle/working/patchcore/bottle/restore-EXP-002")
RESTORE_DIR.mkdir(parents=True, exist_ok=True)

restore_report = {
    "run_id": "patchcore-bottle-restore-exp002",
    "source_run_id": report["run_id"],
    "experiment_id": "EXP-002",
    "source_kaggle_version_url": SOURCE_KAGGLE_URL,
    "git_commit": source_commit,
    "git_tag": EXPECTED_TAG,
    "restoration_device": "cpu",
    "checkpoint_sha256": actual_checkpoint_sha256,
    "config_sha256": config_sha256,
    "manifest_sha256": manifest_sha256,
    "memory_bank_shape": list(restored_model.memory_bank.shape),
    "validation_image": sample["image_path"],
    "python_version": sys.version.split()[0],
    "versions": actual_versions,
    "gpu_reference_score": float(expected_score.item()),
    "cpu_restored_score": float(actual_score.item()),
    "anomaly_map_shape": list(actual_map.shape),
    "score_absolute_difference": score_abs_diff,
    "map_max_absolute_difference": map_max_diff,
    "map_mean_absolute_difference": map_mean_diff,
    "map_max_error_relative_to_range": map_relative_range_error,
    "tolerances": {
        "score_absolute": SCORE_ABS_TOL,
        "map_absolute": MAP_ABS_TOL,
        "map_relative_to_range": MAP_RANGE_REL_TOL
    },
    "strict_state_dict_loading": True,
    "memory_bank_recomputed": False,
    "restoration_status": "PASS"
}

restore_report_path = RESTORE_DIR / "restore_report.json"
restore_report_path.write_text(
    json.dumps(restore_report, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8"
)

print("=== RESTAURATION EXP-002 ===")
print("Rapport:", restore_report_path)
print("Checkpoint SHA-256:", restore_report["checkpoint_sha256"])
print("Memory Bank:", restore_report["memory_bank_shape"])
print("Statut:", restore_report["restoration_status"])
print("Preuves de restauration enregistrées : OK")